In [2]:
import pandas as pd
import geopandas as gpd
import sys
import os
import numpy as np

import openmatrix as omx

import openmatrix
openmatrix.numpy = np

In [3]:
landuse_dir = r"C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\land_use_prep"
population_dir = r"C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\synthetic_population"
skim_dir = r"C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\skim_prep"

## subarea land use 

In [4]:
zone_file = gpd.read_file(os.path.join(landuse_dir, "zonal", "shp", "CTPS_TDM23_TAZ_2017g_v202303.shp"))
landuse_file = pd.read_csv(os.path.join(landuse_dir, "processed", "land_use.csv"))

In [5]:
downtown_taz_ids = zone_file[zone_file["ring"].isin([0,1])].taz_id.to_list()

In [6]:
landuse_file = landuse_file[landuse_file["TAZ"].isin(downtown_taz_ids)]

In [6]:
landuse_file.to_csv(os.path.join(landuse_dir, "processed", "land_use_subarea.csv"), index=False)

## subarea population

In [7]:
hh_df = pd.read_csv(os.path.join(population_dir, "processed", "households.csv"))
pop_df = pd.read_csv(os.path.join(population_dir, "processed", "persons.csv"))

In [9]:
hh_df = hh_df[hh_df["TAZ"].isin(downtown_taz_ids)]

In [12]:
pop_df = pop_df[pop_df["household_id"].isin(hh_df["household_id"])]

In [18]:
hh_df.to_csv(os.path.join(population_dir, "processed", "households_subarea.csv"), index=False)
pop_df.to_csv(os.path.join(population_dir, "processed", "persons_subarea.csv"), index=False)

## subarea skims

In [7]:
skims = [
    "hwy_am_pm",
    "hwy_md_ev",
    "nm_daily",
    "tw_am_pm",
    "tw_md_ev",
    "ta_am_pm",
    "ta_md_ev",
]

mapping_name = "ID"

In [8]:
for skim in skims:
    input_omx = os.path.join(skim_dir, "processed", f"{skim}.omx")
    output_omx = os.path.join(skim_dir, "processed", "subarea", f"{skim}_subarea.omx")

    print(f"process {input_omx}")

    with omx.open_file(input_omx, "r") as matrix_in:
        mapping = matrix_in.mapping(mapping_name)
        zones = [z for z in downtown_taz_ids if z in mapping]
        locations = [mapping[z] for z in zones]
        
        zones = np.array(zones, dtype=np.int32)
        locations = np.array(locations)

        with omx.open_file(output_omx, "w") as matrix_out:
            matrix_out.create_mapping(mapping_name, zones)

            for matrix_name in matrix_in.list_matrices():
                data = matrix_in[matrix_name][:]
                subarea_data = data[np.ix_(locations,locations)]
                matrix_out[matrix_name] = subarea_data

                print(f"save matrix {matrix_name}")
    print(f"create {output_omx}")

process C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\skim_prep\processed\hwy_am_pm.omx
save matrix da_time__AM
save matrix da_time__PM
save matrix da_toll__AM
save matrix da_toll__PM
save matrix dist__AM
save matrix dist__PM
save matrix rs_fare__AM
save matrix rs_fare__PM
save matrix sr_time__AM
save matrix sr_time__PM
save matrix sr_toll__AM
save matrix sr_toll__PM
create C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\skim_prep\processed\subarea\hwy_am_pm_subarea.omx
process C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\skim_prep\processed\hwy_md_ev.omx
save matrix da_time__EV
save matrix da_time__MD
save matrix da_toll__EV
save matrix da_toll__MD
save matrix dist__EV
save matrix dist__MD
save matrix rs_fare__EV
save matrix rs_fare__MD
save matrix sr_time__EV
save matrix sr_time__MD
save matrix sr_toll__EV
save matrix sr_toll__MD
create C:\Users\USYS671257\OneDrive - WSP O365\42_Boston_CTPS\skim_prep\processed\subarea\hwy_md_ev_subarea.omx
process C:\Users\U